# Reconhecimento de Locutor — Comparação de Métodos

Experimento exploratório: dado um trecho de voz, dizer **quem fala** (identificação,
closed-set) comparando classificadores simples sobre as mesmas features (MFCC 39-dim).

> ⚠️ **Amostra pequena**: só 2 locutores (A e B). Serve para comparar métodos e montar os
> gráficos — não como validação final.


## 1. Ideia

Cada áudio é picado em segmentos de 2 s; cada segmento é uma amostra. Comparamos métodos
**vetoriais** (1 vetor médio por segmento) e **por frames** (todos os frames MFCC do segmento),
sempre no mesmo split treino/teste.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import librosa
import scipy.signal as signal

from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 4), 'axes.titlesize': 12,
                     'axes.labelsize': 10, 'font.size': 10})

AUDIO_DIR = Path('audio_exemplo')

# Parâmetros herdados da EDA
TARGET_SR = 16000
VOICE_LOW, VOICE_HIGH = 80, 8000
N_FFT, HOP_LENGTH = 1024, 256
N_MFCC = 13            # 13 MFCC + 13 Δ + 13 ΔΔ = 39 dimensões/frame
SEG_DUR = 2.0          # duração dos segmentos de análise
SEEDS = 15             # repetições Monte Carlo da avaliação

# Descoberta automática da base em audio_exemplo/.
# Convenção nova:  locutor_<nome>_<sessao>.wav  -> sessões do mesmo locutor são agrupadas
# Compatível com a base antiga:                            audio_<nome>.wav/.ogg
import re
LOCUTOR_RE = re.compile(r'^locutor_(?P<nome>.+?)_(?P<sessao>s\d+)$')
LEGACY_RE  = re.compile(r'^audio_(?P<nome>.+)$')

SPEAKER_FILES = {}
for f in sorted(AUDIO_DIR.iterdir()):
    if f.suffix.lower() not in ('.wav', '.ogg', '.mp3', '.flac'):
        continue
    m = LOCUTOR_RE.match(f.stem)
    if m:
        SPEAKER_FILES.setdefault(m.group('nome'), []).append(f)
        continue
    m = LEGACY_RE.match(f.stem)
    if m:
        SPEAKER_FILES.setdefault(m.group('nome'), []).append(f)
        continue
    SPEAKER_FILES.setdefault(f.stem, []).append(f)

for sp in SPEAKER_FILES:
    SPEAKER_FILES[sp] = sorted(SPEAKER_FILES[sp])

if not SPEAKER_FILES:
    raise SystemExit(f'Nenhum áudio em {AUDIO_DIR}/')

print('Locutores descobertos:', {sp: len(v) for sp, v in SPEAKER_FILES.items()})
print('Setup OK — parâmetros:', dict(TARGET_SR=TARGET_SR, N_MFCC=39, SEG_DUR=SEG_DUR))


## 2. Pré-processamento e Extração de MFCC

Mesma cadeia da EDA: **subamostragem 16 kHz → filtro passa-banda 80–8000 Hz → MFCC 39-dim**.
Arquivos do mesmo locutor (`locutor_<nome>_s*.wav`) são concatenados para dar mais dados.


In [ ]:
def preprocess(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
    nyq = TARGET_SR / 2.0
    b, a = signal.butter(5, [VOICE_LOW / nyq, min(VOICE_HIGH / nyq, 0.99)], btype='bandpass')
    y = signal.filtfilt(b, a, y)
    return y.astype(np.float32)

def extract_mfccs(y):
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=N_MFCC,
                                n_fft=N_FFT, hop_length=HOP_LENGTH)
    d1 = librosa.feature.delta(mfcc)
    d2 = librosa.feature.delta(mfcc, order=2)
    return np.vstack([mfcc, d1, d2])   # (39, T)

signals = {}
mfccs = {}
for sp, files in SPEAKER_FILES.items():
    y = np.concatenate([preprocess(p) for p in files])
    signals[sp] = y
    mfccs[sp] = extract_mfccs(y)
    print(f'{sp:7s}  duração total={len(y)/TARGET_SR:6.2f}s  frames MFCC={mfccs[sp].shape[1]}  dim={mfccs[sp].shape[0]}')


## 3. Segmentação temporal

Audio contínuo é picado em **segmentos de 2s**. Cada segmento é uma "amostra" independente no nosso
conjunto de dados (mimicando o uso real: o sistema analisa janelas de voz e decide quem está falando).


In [ ]:
def make_segments(mfcc, seg_dur=SEG_DUR):
    frames_per_seg = int(round(seg_dur * TARGET_SR / HOP_LENGTH))
    n = mfcc.shape[1]
    return [mfcc[:, i:i + frames_per_seg]
            for i in range(0, n - frames_per_seg + 1, frames_per_seg)]

segments, speaker_ids = [], []
for sp, m in mfccs.items():
    segs = make_segments(m)
    segments.extend(segs)
    speaker_ids.extend([sp] * len(segs))
    print(f'{sp:7s} -> {len(segs)} segmentos de {SEG_DUR}s')

speaker_ids = np.array(speaker_ids)
SPEAKERS = sorted(np.unique(speaker_ids))
print(f'Total: {len(segments)} segmentos | locutores: {SPEAKERS}')

# Normalização do cepstrum (CMN): remove desvio de canal/microfone por segmento.
# Aplicada no método baseado em frames (GMM).
def cmn(seg):
    return seg - seg.mean(axis=1, keepdims=True)

segments_cmn = [cmn(s) for s in segments]

# Representação vetorial por segmento (média temporal dos 39 MFCC) p/ métodos vetoriais.
mean_vecs = np.array([s.mean(axis=1) for s in segments])
print(f'mean_vecs shape: {mean_vecs.shape}')


## 4. Divisão treino/teste (estratificada)

Como os segmentos de cada locutor são contínuos (frames correlacionados), avaliamos de forma robusta
com **split aleatório repetido (Monte Carlo, `SEEDS` iterações)** e reportamos média±desvio da acurácia.


In [ ]:
def make_split(seed):
    tr, te = train_test_split(np.arange(len(segments)), test_size=0.3,
                              stratify=speaker_ids, random_state=seed)
    return tr, te

# Exemplo visual do split (seed 0)
tr, te = make_split(0)
print(f'seed=0 -> treino {len(tr)}, teste {len(te)}')
for sp in SPEAKERS:
    n_tr = (speaker_ids[tr] == sp).sum()
    n_te = (speaker_ids[te] == sp).sum()
    print(f'  {sp}: treino={n_tr}  teste={n_te}')


## 5. Métodos comparados

| # | Método | Tipo | Ideia |
|---|--------|------|-------|
| 1 | **Centroide + cosseno** | vetorial | Protótipo = média dos segmentos de treino; classifica pelo cosseno mais próximo. |
| 2 | **k-NN (k=3)** | vetorial | Voto dos 3 segmentos de treino mais próximos (cosseno). |
| 3 | **GMM por locutor** | frames | 1 GMM diagonal (4 gaussianas) por locutor; vence a maior verossimilhança. |
| 4 | **SVM (RBF)** | vetorial | Hiperplano de margem máxima no vetor médio. |
| 5 | **MLP (64)** | vetorial | Rede pequena (1 camada oculta) no vetor médio. |


In [ ]:
# ---------- Modelos (interface fit/predict) ----------
def frames_of(X, cls, y):
    return np.concatenate([seg for seg, lab in zip(X, y) if lab == cls], axis=1).T


class CentroidModel:
    def fit(self, X, y):
        self.classes_ = np.unique(np.asarray(y))
        self.W_ = np.array([np.asarray(X)[np.asarray(y) == c].mean(0) for c in self.classes_])
        return self
    def predict(self, X):
        X = np.asarray(X); X = X / np.linalg.norm(X, axis=1)[:, None]
        W = self.W_ / np.linalg.norm(self.W_, axis=1)[:, None]
        return np.array([self.classes_[i] for i in np.argmax(X @ W.T, axis=1)])


class GMMModel:
    def __init__(self, n_components=4): self.n_components = n_components
    def fit(self, X, y):
        self.models_ = {}
        for c in np.unique(np.asarray(y)):
            self.models_[c] = GaussianMixture(self.n_components, covariance_type='diag',
                                              random_state=0).fit(frames_of(X, c, np.asarray(y)))
        return self
    def predict(self, X):
        preds = []
        for seg in X:
            scores = {c: m.score_samples(seg.T).mean() for c, m in self.models_.items()}
            preds.append(max(scores.items(), key=lambda kv: kv[1])[0])
        return np.array(preds)


METHODS = [
    ('1. Centroide + cosseno', CentroidModel(), 'mean'),
    ('2. k-NN (k=3, cosseno)', KNeighborsClassifier(n_neighbors=3, weights='distance', metric='cosine'), 'mean'),
    ('3. GMM por locutor (4 gauss)', GMMModel(n_components=4), 'frame'),
    ('4. SVM (RBF)', SVC(C=1.0, kernel='rbf', gamma='scale'), 'mean'),
    ('5. MLP (64 neuronios)', MLPClassifier(hidden_layer_sizes=(64,), max_iter=3000, random_state=0), 'mean'),
]
DATA = {'mean': (mean_vecs, speaker_ids), 'frame': (segments_cmn, speaker_ids)}
print(f'{len(METHODS)} metodos registrados.')


## 6. Avaliação — Identificação de Locutor (Monte Carlo)

Para cada um dos `SEEDS` splits: treina o modelo, classifica os segmentos de teste, mede acurácia.


In [ ]:
results = defaultdict(list)
for seed in range(SEEDS):
    tr, te = make_split(seed)
    for name, model, kind in METHODS:
        X, y = DATA[kind]
        X_tr = X[tr] if kind == 'mean' else [X[i] for i in tr]
        X_te = X[te] if kind == 'mean' else [X[i] for i in te]
        model.fit(X_tr, y[tr])
        pred = model.predict(X_te)
        results[name].append(accuracy_score(y[te], pred))

print(f'{"Método":28s} {"Acurácia média":>16s} {"±":>2s} {"Desvio":>6s}')
print('-' * 58)
summary = []
for name, _, _ in METHODS:
    acc = np.mean(results[name]); std = np.std(results[name])
    summary.append((name, acc, std))
    print(f'{name:28s} {acc*100:14.1f}% ± {std*100:5.1f}%')


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
names = [s[0] for s in summary]
means = [s[1]*100 for s in summary]
stds  = [s[2]*100 for s in summary]
colors = plt.cm.viridis(np.linspace(0.25, 0.95, len(names)))
bars = ax.barh(names[::-1], means[::-1], xerr=stds[::-1], color=colors[::-1],
               capsize=4, edgecolor='black', linewidth=0.5)
best = np.argmax(means)
bars[len(names)-1-best].set_edgecolor('crimson'); bars[len(names)-1-best].set_linewidth(2.5)
ax.set_xlabel('Acurácia de identificação (%)')
ax.set_title(f'Comparação de métodos — média ± desvio sobre {SEEDS} splits (closed-set)', fontsize=13)
for i, (m, s) in enumerate(zip(means[::-1], stds[::-1])):
    ax.text(m + 1, i, f'{m:.1f}% ± {s:.1f}', va='center', fontsize=9)
ax.set_xlim(0, 110)
plt.tight_layout(); plt.show()


## 7. Dispersão entre splits (boxplot)

A barra mostra média ± desvio; o boxplot mostra a distribuição real da acurácia nos
`SEEDS` splits — com base pequena, a dispersão importa tanto quanto a média.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
data = [np.array(results[name]) * 100 for name, _, _ in METHODS]
bp = ax.boxplot(data, vert=False, patch_artist=True)
ax.set_yticklabels([name for name, _, _ in METHODS])
ax.set_xlabel('Acuracia de identificacao (%)')
ax.set_title(f'Dispersao da acuracia sobre {SEEDS} splits')
plt.tight_layout(); plt.show()


## 8. Matrizes de confusão (todos os métodos, seed 0)

Quem confunde com quem, método por método, no mesmo split fixo.


In [ ]:
tr, te = make_split(0)
n = len(METHODS)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.2))
for ax, (name, model, kind) in zip(np.atleast_1d(axes).flat, METHODS):
    X, y = DATA[kind]
    X_tr = X[tr] if kind == 'mean' else [X[i] for i in tr]
    X_te = X[te] if kind == 'mean' else [X[i] for i in te]
    model.fit(X_tr, y[tr])
    pred = model.predict(X_te)
    cm = confusion_matrix(y[te], pred, labels=SPEAKERS)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(SPEAKERS))); ax.set_xticklabels(SPEAKERS, rotation=30)
    ax.set_yticks(range(len(SPEAKERS))); ax.set_yticklabels(SPEAKERS)
    ax.set_title(f'{name}\nacc={accuracy_score(y[te], pred)*100:.1f}%', fontsize=9)
    for i in range(len(SPEAKERS)):
        for j in range(len(SPEAKERS)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.suptitle('Matrizes de confusao por metodo (seed 0)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()
